In [ ]:
%load_ext autoreload
%autoreload 2

Declaration of parameters (you must also add a tag for this cell - parameters)

In [ ]:
# specify substep parameters for interactive run
# this cell will be replaced during job run with the parameters from json within params subfolder
substep_params={   
    "threshold_iou" : 0.5,
    "threshold_MAp05": 0.01
}

In [ ]:
# load pipeline and step parameters - do not edit
from sinara.substep import get_pipeline_params, get_step_params
pipeline_params = get_pipeline_params(pprint=True)
step_params = get_step_params(pprint=True)

In [ ]:
# define substep interface
from sinara.substep import NotebookSubstep, ENV_NAME, PIPELINE_NAME, ZONE_NAME, STEP_NAME, RUN_ID, ENTITY_NAME, ENTITY_PATH, SUBSTEP_NAME

substep = NotebookSubstep(pipeline_params, step_params, substep_params)

substep.interface(    
    tmp_inputs =
    [
        { ENTITY_NAME: "test_dataset" }, # ground-true test dataset
        { ENTITY_NAME: "predict_data" }, # predicted test dataset
    ]
)

substep.print_interface_info()

substep.exit_in_visualize_mode()

### Load ground truth and predicted test dataset markups

In [ ]:
import os.path as osp
from utils.coco import load as coco_load

tmp_inputs = substep.tmp_inputs()

# reading ground-true test dataset markup 
eval_gt_file = osp.join(osp.join(tmp_inputs.test_dataset, "test_coco_annotations.json"))
eval_gt = coco_load(eval_gt_file)

# reading predicted test dataset markup 
eval_dt_file = osp.join(tmp_inputs.predict_data, 'eval_dt_torch.json')
eval_dt = coco_load(eval_dt_file)

iouType = eval_dt['info'].get('iouType', 'bbox')

### Evaluate the test dataset 

#### Eval Precision-Recall Curve

In [ ]:
from pycocotools.coco import COCO
from utils.extra.curves import Curves
from utils.coco.encoder import encode_dict_to_numpy

prepared_coco_in_dict = encode_dict_to_numpy(eval_gt)

_curves = []
prepared_anns = encode_dict_to_numpy(eval_dt['annotations'])

cocoGt = COCO(eval_gt_file)
cocoDt = cocoGt.loadRes(prepared_anns)

cur = Curves(
    cocoGt, 
    cocoDt, 
    iou_tresh=substep_params['threshold_iou'], 
    recall_count=1000, 
    iouType=iouType,
    # useCats=True,
)

_curve = cur.build_curve('category')
cur.plot_pre_rec(curves=_curve, plotly_backend=True)

#### Eval Average Precision, Recall Metrics

In [ ]:
from pycocotools.cocoeval import COCOeval

annType = eval_dt['info'].get('iouType', 'bbox')
imgIds=sorted(cocoGt.getImgIds())

# running evaluation
cocoEval = COCOeval(cocoGt,cocoDt,annType)
cocoEval.params.imgIds  = imgIds
cocoEval.evaluate()
cocoEval.accumulate()
cocoEval.summarize()

### Check model by metric mAP@0.5 IoU

In [ ]:
threshold_MAp05 = substep_params["threshold_MAp05"]

map_05 = cocoEval.stats[1]
print(f"mAP@0.5 = {map_05}")
assert map_05 > threshold_MAp05, f"The calculated MAp metric on the test dataset is less than the acceptable value <{threshold_MAp05}"